In [6]:
from huggingface_hub import login
login(token="")

In [7]:
try:
    import google.colab  # type: ignore
    from google.colab import output

    COLAB = True
    %pip install sae-lens transformer-lens sae-dashboard
except:
    COLAB = False
    from IPython import get_ipython  # type: ignore

    ipython = get_ipython()
    assert ipython is not None
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

import os
import torch
from tqdm import tqdm
import plotly.express as px
import pandas as pd

torch.set_grad_enabled(False)

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Device: cuda:0


In [9]:
from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory

# TODO: Make this nicer.
df = pd.DataFrame.from_records(
    {k: v.__dict__ for k, v in get_pretrained_saes_directory().items()}
).T
df.drop(
    columns=[
        "expected_var_explained",
        "expected_l0",
        "config_overrides",
        "conversion_func",
    ],
    inplace=True,
)

In [10]:
df.head(2) # Each row is a "release" which has multiple SAEs which may have different configs / match different hook points in a model.

,release,repo_id,model,saes_map,neuronpedia_id
deepseek-r1-distill-llama-8b-qresearch,deepseek-r1-distill-llama-8b-qresearch,qresearch/DeepSeek-R1-Distill-Llama-8B-SAE-l19,deepseek-ai/DeepSeek-R1-Distill-Llama-8B,{'blocks.19.hook_resid_post': 'DeepSeek-R1-Dis...,{'blocks.19.hook_resid_post': 'deepseek-r1-dis...
gemma-2-2b-res-matryoshka-dc,gemma-2-2b-res-matryoshka-dc,chanind/gemma-2-2b-batch-topk-matryoshka-saes-...,gemma-2-2b,{'blocks.0.hook_resid_post': 'standard/blocks....,"{'blocks.0.hook_resid_post': None, 'blocks.1.h..."


In [11]:
filtered_df = df[df['model'].str.contains('gemma-2-9b', na=False)]
filtered_df

,release,repo_id,model,saes_map,neuronpedia_id
gemma-2-9b-res-matryoshka-dc,gemma-2-9b-res-matryoshka-dc,chanind/gemma-2-9b-batch-topk-matryoshka-saes-...,gemma-2-9b,{'blocks.0.hook_resid_post': 'blocks.0.hook_re...,"{'blocks.0.hook_resid_post': None, 'blocks.1.h..."
gemma-scope-9b-it-res,gemma-scope-9b-it-res,google/gemma-scope-9b-it-res,gemma-2-9b,{'layer_20/width_131k/average_l0_13': 'layer_2...,"{'layer_20/width_131k/average_l0_13': None, 'l..."
gemma-scope-9b-it-res-canonical,gemma-scope-9b-it-res-canonical,google/gemma-scope-9b-it-res,gemma-2-9b-it,{'layer_9/width_16k/canonical': 'layer_9/width...,{'layer_9/width_16k/canonical': 'gemma-2-9b-it...
gemma-scope-9b-pt-att,gemma-scope-9b-pt-att,google/gemma-scope-9b-pt-att,gemma-2-9b,{'layer_0/width_16k/average_l0_12': 'layer_0/w...,"{'layer_0/width_16k/average_l0_12': None, 'lay..."
gemma-scope-9b-pt-att-canonical,gemma-scope-9b-pt-att-canonical,google/gemma-scope-9b-pt-att,gemma-2-9b,{'layer_0/width_16k/canonical': 'layer_0/width...,{'layer_0/width_16k/canonical': 'gemma-2-9b/0-...
gemma-scope-9b-pt-mlp,gemma-scope-9b-pt-mlp,google/gemma-scope-9b-pt-mlp,gemma-2-9b,{'layer_0/width_16k/average_l0_6': 'layer_0/wi...,"{'layer_0/width_16k/average_l0_6': None, 'laye..."
gemma-scope-9b-pt-mlp-canonical,gemma-scope-9b-pt-mlp-canonical,google/gemma-scope-9b-pt-mlp,gemma-2-9b,{'layer_0/width_16k/canonical': 'layer_0/width...,{'layer_0/width_16k/canonical': 'gemma-2-9b/0-...
gemma-scope-9b-pt-res,gemma-scope-9b-pt-res,google/gemma-scope-9b-pt-res,gemma-2-9b,{'embedding/width_4k/average_l0_14': 'embeddin...,"{'embedding/width_4k/average_l0_14': None, 'em..."
gemma-scope-9b-pt-res-canonical,gemma-scope-9b-pt-res-canonical,google/gemma-scope-9b-pt-res,gemma-2-9b,{'layer_0/width_16k/canonical': 'layer_0/width...,{'layer_0/width_16k/canonical': 'gemma-2-9b/0-...


In [12]:
filtered_df[filtered_df["release"]=="gemma-scope-9b-it-res-canonical"].saes_map.iloc[0]

{'layer_9/width_16k/canonical': 'layer_9/width_16k/average_l0_88',
 'layer_20/width_16k/canonical': 'layer_20/width_16k/average_l0_91',
 'layer_31/width_16k/canonical': 'layer_31/width_16k/average_l0_76',
 'layer_9/width_131k/canonical': 'layer_9/width_131k/average_l0_121',
 'layer_20/width_131k/canonical': 'layer_20/width_131k/average_l0_81',
 'layer_31/width_131k/canonical': 'layer_31/width_131k/average_l0_109'}

# Loading a pretrained Sparse Autoencoder

In [13]:
# from transformer_lens import HookedTransformer
from sae_lens import SAE, HookedSAETransformer

model = HookedSAETransformer.from_pretrained("gemma-2-9b-it", device=device)

# the cfg dict is returned alongside the SAE since it may contain useful information for analysing the SAE
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-9b-it-res-canonical",  # <- Release name
    sae_id="layer_31/width_131k/canonical",  # <- SAE id (not always a hook point!)
    device=device,
)

Loading checkpoint shards: 100%|███████████████████████████████████████| 4/4 [02:42<00:00, 40.50s/it]


Loaded pretrained model gemma-2-9b-it into HookedTransformer


In [15]:
print(sae.cfg.__dict__)

{'architecture': 'jumprelu', 'd_in': 3584, 'd_sae': 131072, 'activation_fn_str': 'relu', 'apply_b_dec_to_input': False, 'finetuning_scaling_factor': False, 'context_size': 1024, 'model_name': 'gemma-2-9b-it', 'hook_name': 'blocks.31.hook_resid_post', 'hook_layer': 31, 'hook_head_index': None, 'prepend_bos': True, 'dataset_path': 'monology/pile-uncopyrighted', 'dataset_trust_remote_code': True, 'normalize_activations': None, 'dtype': 'float32', 'device': 'cuda:0', 'sae_lens_training_version': None, 'activation_fn_kwargs': {}, 'neuronpedia_id': 'gemma-2-9b-it/31-gemmascope-res-131k', 'model_from_pretrained_kwargs': {}, 'seqpos_slice': (None,)}


### The Contrast Pairs

In [21]:
prompt = [
    "Jack failed his math exam. He felt deeply discouraged, believing he was incapable of achieving anything meaningful.",
    "Jack failed his math exam. He felt motivated, viewing the experience as an opportunity to improve himself.",
]

_, cache = model.run_with_cache_with_saes(prompt, saes=[sae])
print([(k, v.shape) for k, v in cache.items() if "sae" in k])

# feature_activation_df = pd.DataFrame(
#     cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][0, -1, :].cpu().numpy(),
#     index=[f"feature_{i}" for i in range(sae.cfg.d_sae)],
# )
# feature_activation_df.columns = ["heavens_and_the"]
# feature_activation_df["cat_and_the"] = (
#     cache["blocks.7.hook_resid_pre.hook_sae_acts_post"][1, -1, :].cpu().numpy()
# )
# feature_activation_df["diff"] = (
#     feature_activation_df["heavens_and_the"] - feature_activation_df["cat_and_the"]
# )

# fig = px.line(
#     feature_activation_df,
#     title="Feature activations for the prompt",
#     labels={"index": "Feature", "value": "Activation"},
# )

# # hide the x-ticks
# fig.update_xaxes(showticklabels=False)
# fig.show()

[('blocks.31.hook_resid_post.hook_sae_input', torch.Size([2, 21, 3584])), ('blocks.31.hook_resid_post.hook_sae_acts_pre', torch.Size([2, 21, 131072])), ('blocks.31.hook_resid_post.hook_sae_acts_post', torch.Size([2, 21, 131072])), ('blocks.31.hook_resid_post.hook_sae_recons', torch.Size([2, 21, 3584])), ('blocks.31.hook_resid_post.hook_sae_output', torch.Size([2, 21, 3584]))]


In [22]:
cache

ActivationCache with keys ['hook_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_rot_q', 'blocks.0.attn.hook_rot_k', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.ln1_post.hook_scale', 'blocks.0.ln1_post.hook_normalized', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_pre_linear', 'blocks.0.mlp.hook_post', 'blocks.0.ln2_post.hook_scale', 'blocks.0.ln2_post.hook_normalized', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_rot_q', 'blocks.1.attn.hook_rot_k', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pat

In [23]:
cache["blocks.31.hook_resid_post.hook_sae_acts_post"].shape

torch.Size([2, 21, 131072])

In [24]:
# let's look at the biggest features in terms of absolute difference

diff = (
    cache["blocks.31.hook_resid_post.hook_sae_acts_post"][1, -1, :].cpu()
    - cache["blocks.31.hook_resid_post.hook_sae_acts_post"][0, -1, :].cpu()
    # cache["blocks.20.hook_resid_post.hook_sae_acts_post"][0, -1, :].cpu()
)
vals, inds = torch.topk(torch.abs(diff), 5)
for val, ind in zip(vals, inds):
    print(f"Feature {ind} had a difference of {val:.2f}")

Feature 78648 had a difference of 72.20
Feature 51034 had a difference of 48.63
Feature 111832 had a difference of 34.36
Feature 68503 had a difference of 33.18
Feature 51962 had a difference of 31.45


In [32]:
from IPython.display import IFrame


html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"


def get_dashboard_html(sae_release="gemma-2-9b", sae_id="31-gemmascope-res-16k", feature_idx=0):
    return html_template.format(sae_release, sae_id, feature_idx)

html = get_dashboard_html(
    sae_release="gemma-2-9b-it", sae_id="31-gemmascope-res-131k", feature_idx=51034
)
IFrame(html, width=1200, height=600)

In [33]:
print(sae.cfg.dataset_path)

monology/pile-uncopyrighted


In [27]:
from datasets import load_dataset

hf_dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)

In [28]:
# instantiate an object to hold activations from a dataset
from sae_lens import ActivationsStore

# a convenient way to instantiate an activation store is to use the from_sae method
activation_store = ActivationsStore.from_sae(
    model=model,
    sae=sae,
    streaming=True,
    dataset=hf_dataset,
    # fairly conservative parameters here so can use same for larger
    # models without running out of memory.
    store_batch_size_prompts=8,
    train_batch_size_tokens=4096,
    n_batches_in_buffer=32,
    device=device,
)

/home/bangzhao/.local/lib/python3.12/site-packages/sae_lens/training/activations_store.py:301: UserWarning: Dataset is not tokenized. Pre-tokenizing will improve performance and allows for more control over special tokens. See https://jbloomaus.github.io/SAELens/training_saes/#pretokenizing-datasets for more info.
  warnings.warn(


## Feature Steering

In [36]:
from tqdm import tqdm
from functools import partial


def find_max_activation(model, sae, activation_store, feature_idx, num_batches=50):
    """
    Find the maximum activation for a given feature index. This is useful for
    calibrating the right amount of the feature to add.
    """
    max_activation = 0.0

    pbar = tqdm(range(num_batches))
    for _ in pbar:
        tokens = activation_store.get_batch_tokens()

        _, cache = model.run_with_cache(
            tokens,
            stop_at_layer=sae.cfg.hook_layer + 1,
            names_filter=[sae.cfg.hook_name],
        )
        sae_in = cache[sae.cfg.hook_name]
        feature_acts = sae.encode(sae_in).squeeze()

        feature_acts = feature_acts.flatten(0, 1)
        batch_max_activation = feature_acts[:, feature_idx].max().item()
        max_activation = max(max_activation, batch_max_activation)

        pbar.set_description(f"Max activation: {max_activation:.4f}")

    return max_activation


def steering(
    activations, hook, steering_strength=1.0, steering_vector=None, max_act=1.0
):
    # Note if the feature fires anyway, we'd be adding to that here.
    return activations + max_act * steering_strength * steering_vector


def generate_with_steering(
    model,
    sae,
    prompt,
    steering_feature,
    max_act,
    steering_strength=1.0,
    max_new_tokens=95,
):
    input_ids = model.to_tokens(prompt, prepend_bos=sae.cfg.prepend_bos)

    steering_vector = sae.W_dec[steering_feature].to(model.cfg.device)

    steering_hook = partial(
        steering,
        steering_vector=steering_vector,
        steering_strength=steering_strength,
        max_act=max_act,
    )

    # standard transformerlens syntax for a hook context for generation
    with model.hooks(fwd_hooks=[(sae.cfg.hook_name, steering_hook)]):
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            stop_at_eos=False if device == "mps" else True,
            prepend_bos=sae.cfg.prepend_bos,
        )

    return model.tokenizer.decode(output[0])


# Choose a feature to steer
steering_feature = steering_feature = 78648  # Choose a feature to steer towards

# Find the maximum activation for this feature
max_act = find_max_activation(model, sae, activation_store, steering_feature)
print(f"Maximum activation for feature {steering_feature}: {max_act:.4f}")

# note we could also get the max activation from Neuronpedia (https://www.neuronpedia.org/api-doc#tag/lookup/GET/api/feature/{modelId}/{layer}/{index})

prompt = """Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
"""

normal_text = model.generate(
    prompt,
    max_new_tokens=95,
    stop_at_eos=False if device == "mps" else True,
    prepend_bos=sae.cfg.prepend_bos,
)

print("\nNormal text (without steering):")
print(normal_text)

# Generate text with steering
steered_text = generate_with_steering(
    model, sae, prompt, steering_feature, max_act, steering_strength=2.0
)
print("Steered text:")
print(steered_text)

Max activation: 119.8887: 100%|██████████████████████████████████████| 50/50 [05:31<00:00,  6.63s/it]


Maximum activation for feature 78648: 119.8887


100%|████████████████████████████████████████████████████████████████| 95/95 [00:05<00:00, 16.98it/s]



Normal text (without steering):
Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
 **C) Stay up and lend a listening ear to her friend**

**Explanation:**

While all of the choices have their drawbacks, option C is the most compassionate and helpful. Here's why:

* **A) Ignore the texts and continue sleeping:** This is dismissive and could hurt Regina's friend further. 
* **B) Respond telling her friend to seek professional help:** While professional help can be beneficial, it might come across as insensitive


100%|████████████████████████████████████████████████████████████████| 95/95 [00:05<00:00, 17.13it/s]

Steered text:
<bos>Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
The most effective action for Regina in this scenario is **C) Stay up and lend a listening ear to her friend.** 

Here's why:

* **Empathy and Support:**  When someone is feeling down, especially after a breakup, they need someone to listen and validate their feelings.  

* **Avoidance:** Ignoring the texts (A) would make Regina's friend feel even more alone and unloved.

* **Overly Serious:**


In [37]:
# Experiment with different steering strengths
print("\nExperimenting with different steering strengths:")
for strength in [-4.0, -2.0, -1.0, 0.2, 0.5, 1.0, 2.0, 4.0]:
    steered_text = generate_with_steering(
        model, sae, prompt, steering_feature, max_act, steering_strength=strength
    )
    print(f"\nSteering strength {strength}:")
    print(steered_text)


Experimenting with different steering strengths:


100%|████████████████████████████████████████████████████████████████| 95/95 [00:05<00:00, 17.14it/s]



Steering strength -4.0:
<bos>Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
The most effective action for Regina in this scenario is **C) Stay up and lend a listening ear to her friend.**

Here's why:

* **Unveiling the Importance of Support:**  When someone is experiencing a significant event like a breakup, they need support and someone to listen.  

* **Building Stronger Relationships:**  Responding with care and understanding strengthens the bond between friends.

* **Respecting Her Needs:**  Even if it's


100%|████████████████████████████████████████████████████████████████| 95/95 [00:05<00:00, 16.97it/s]



Steering strength -2.0:
<bos>Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
**C) Stay up and lend a listening ear to her friend**

**Explanation:**

In this situation, Regina's friend needs immediate support and reassurance. 

* **A) Ignore the texts and continue sleeping:** This would be dismissive and could further isolate her friend.
* **B) Respond telling her friend to seek professional help:** While professional help is valuable, it's not the most immediate and helpful response in the middle of the night when


...


Steering strength 1.0:
<bos>Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
**C) Stay up and lend a listening ear to her friend**

Here's why:

* **Emotional Support:**  Regina's friend is clearly hurting and needs someone to listen without judgment.  Staying up to offer support shows empathy and strengthens their friendship.
* **Validation:** Feeling heard and understood can be incredibly comforting during a breakup.  Regina can validate her friend's feelings without offering unsolicited advice.
* **Availability:** Being there for a


100%|████████████████████████████████████████████████████████████████| 95/95 [00:05<00:00, 16.88it/s]



Steering strength 2.0:
<bos>Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
The most effective action is **C) Stay up and lend a listening ear to her friend.**

Here's why:

* **Empathy and Support:**  When someone is feeling down, especially after a breakup, they need someone to listen and validate their feelings.  
* **Availability:** Being there for a friend in a time of need shows you care and are a reliable source of support.
* **Avoidance:**  Options A and D are not


100%|████████████████████████████████████████████████████████████████| 95/95 [00:05<00:00, 16.86it/s]


Steering strength 4.0:
<bos>Scenario: Regina's best friend recently broke up with her longtime partner and is texting Regina in the middle of the night expressing feelings of loneliness.
Question: What is the most effective action for Robert in this scenario?
Choices:
A) Ignore the texts and continue sleeping
B) Respond telling her friend to seek professional help
C) Stay up and lend a listening ear to her friend
D) Suggest her friend to find a new partner

Answer:
The most effective and empathetic response is **C) Stay down and lend a listening ear to her friend.**

Here's why:

* **Downcast and Sad:** Depression often manifests as sadness, sadness, and a feeling of hopelessness. Feeling down is a common symptom of sadness.
* **Feeling Down and Depressed:** Feeling down is a symptom of sadness, which can be a symptom of depression.

Let's look at why the other options are not


We can also do this via the Neuronpedia AP or on the website [here](https://www.neuronpedia.org/steer/). The example below steers for just a few tokens with a feature that does something very specific. Can you work out what it's doing?

In [27]:
import requests
import numpy as np

url = "https://www.neuronpedia.org/api/steer"

payload = {
    # "prompt": "A knight in shining",
    # "prompt": "He had to fight back in self-",
    "prompt": "In the middle of the universe is the galactic",
    # "prompt": "Oh no. We're running on empty. Its time to fill up the car with",
    # "prompt": "Sure, I'm happy to pay. I don't have any cash on me but let me write you a",
    "modelId": "gpt2-small",
    "features": [
        {"modelId": "gpt2-small", "layer": "7-res-jb", "index": 6770, "strength": 8}
    ],
    "temperature": 0.2,
    "n_tokens": 2,
    "freq_penalty": 1,
    "seed": np.random.randint(100),
    "strength_multiplier": 4,
}
headers = {"Content-Type": "application/json"}

response = requests.post(url, json=payload, headers=headers)

print(response.json())

{'STEERED': 'In the middle of the universe is the galactic centre,', 'DEFAULT': 'In the middle of the universe is the galactic center,', 'id': 'cm05iscn10001pbsvlwa7u39v', 'shareUrl': 'https://www.neuronpedia.org/steer/cm05iscn10001pbsvlwa7u39v', 'limit': '299'}


In [55]:
import requests
import numpy as np

url = "https://www.neuronpedia.org/api/steer"

payload = {
    "prompt": 'I wrote a letter to my girlfiend. It said "',
    "modelId": "gpt2-small",
    "features": [
        {"modelId": "gpt2-small", "layer": "7-res-jb", "index": 20115, "strength": 4}
    ],
    "temperature": 0.7,
    "n_tokens": 120,
    "freq_penalty": 1,
    "seed": np.random.randint(100),
    "strength_multiplier": 4,
}
headers = {"Content-Type": "application/json"}

response = requests.post(url, json=payload, headers=headers)

print(response.json())

{'STEERED': 'I wrote a letter to my girlfiend. It said "Dear Assembles Of Life" and I read it. And I know that reading the letter and the message in its entirety, I wonder . . .\n\nIt is for them that we have been called to care, and watch our hearts seize. For God loves us when we are strong; for us that not only of flesh but of spirit must be filled with love;\n\nFor our hearts are full of mercy;\n\nAnd we who are weak shall have strength, and they shall set free: And so together shall they live together: And their lips are upon me, that such', 'DEFAULT': 'I wrote a letter to my girlfiend. It said "Dear Asher,\n\nI have decided to access your files. I am reading the contents and it is in contact with me for some time now.\n\nIt is very traumatic that you have been allowed to access those files because you are a member of the group that has been so hostile toward us. We can only assume that you are an individual and not a criminal defendant or anything like that, but I don\'t believe 

## Feature Ablation

Feature ablation is also worth looking at. In a way, it's a special case of steering where the value of the feature is always zeroed out.

Here we do the following:
1. Use test prompt rather than generate to get more nuance. 
2. attach a hook to the SAE feature activations.
3. 0 out a feature at all positions (we know that the default feature fires at the final position.)
4. Check whether this ablation is more / less effective if we include the error term (info our SAE isn't capturing).

Note that the existence of [The Hydra Effect](https://arxiv.org/abs/2307.15771) can make reasoning about ablation experiments difficult.

In [1]:
from transformer_lens.utils import test_prompt
from functools import partial


def test_prompt_with_ablation(model, sae, prompt, answer, ablation_features):
    def ablate_feature_hook(feature_activations, hook, feature_ids, position=None):
        if position is None:
            feature_activations[:, :, feature_ids] = 0
        else:
            feature_activations[:, position, feature_ids] = 0

        return feature_activations

    ablation_hook = partial(ablate_feature_hook, feature_ids=ablation_features)

    model.add_sae(sae)
    hook_point = sae.cfg.hook_name + ".hook_sae_acts_post"
    model.add_hook(hook_point, ablation_hook, "fwd")

    test_prompt(prompt, answer, model)

    model.reset_hooks()
    model.reset_saes()


# Example usage in a notebook:

# Assume model and sae are already defined

# Choose a feature to ablate

model.reset_hooks(including_permanent=True)
prompt = "In the beginning, God created the heavens and the"
answer = "earth"
test_prompt(prompt, answer, model)


# Generate text with feature ablation
print("Test Prompt with feature ablation and no error term")
ablation_feature = 16873  # Replace with any feature index you're interested in. We use the religion feature
sae.use_error_term = False
test_prompt_with_ablation(model, sae, prompt, answer, ablation_feature)

print("Test Prompt with feature ablation and error term")
ablation_feature = 16873  # Replace with any feature index you're interested in. We use the religion feature
sae.use_error_term = True
test_prompt_with_ablation(model, sae, prompt, answer, ablation_feature)

/opt/anaconda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-10 15:25:30.424761: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752161130.440994 1758928 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752161130.445314 1758928 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752161130.458129 1758928 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

[2025-07-10 15:25:35,709] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/opt/anaconda/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/opt/anaconda/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda-12.4/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/opt/anaconda/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda-12.4/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/opt/anaconda/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda-12.4/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/opt/anaconda/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda-12.4/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/opt/anaconda/compiler_compat/ld: /usr/local/cuda-12.4/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/opt/anaconda/compiler_compat/ld: /usr/local/cuda-12.4/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@C

[2025-07-10 15:25:38,325] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


NameError: name 'model' is not defined

# Feature Attribution

In [57]:
from dataclasses import dataclass
from functools import partial
from typing import Any, Literal, NamedTuple, Callable

import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer
from transformer_lens.hook_points import HookPoint


class SaeReconstructionCache(NamedTuple):
    sae_in: torch.Tensor
    feature_acts: torch.Tensor
    sae_out: torch.Tensor
    sae_error: torch.Tensor


def track_grad(tensor: torch.Tensor) -> None:
    """wrapper around requires_grad and retain_grad"""
    tensor.requires_grad_(True)
    tensor.retain_grad()


@dataclass
class ApplySaesAndRunOutput:
    model_output: torch.Tensor
    model_activations: dict[str, torch.Tensor]
    sae_activations: dict[str, SaeReconstructionCache]

    def zero_grad(self) -> None:
        """Helper to zero grad all tensors in this object."""
        self.model_output.grad = None
        for act in self.model_activations.values():
            act.grad = None
        for cache in self.sae_activations.values():
            cache.sae_in.grad = None
            cache.feature_acts.grad = None
            cache.sae_out.grad = None
            cache.sae_error.grad = None


def apply_saes_and_run(
    model: HookedTransformer,
    saes: dict[str, SAE],
    input: Any,
    include_error_term: bool = True,
    track_model_hooks: list[str] | None = None,
    return_type: Literal["logits", "loss"] = "logits",
    track_grads: bool = False,
) -> ApplySaesAndRunOutput:
    """
    Apply the SAEs to the model at the specific hook points, and run the model.
    By default, this will include a SAE error term which guarantees that the SAE
    will not affect model output. This function is designed to work correctly with
    backprop as well, so it can be used for gradient-based feature attribution.

    Args:
        model: the model to run
        saes: the SAEs to apply
        input: the input to the model
        include_error_term: whether to include the SAE error term to ensure the SAE doesn't affect model output. Default True
        track_model_hooks: a list of hook points to record the activations and gradients. Default None
        return_type: this is passed to the model.run_with_hooks function. Default "logits"
        track_grads: whether to track gradients. Default False
    """

    fwd_hooks = []
    bwd_hooks = []

    sae_activations: dict[str, SaeReconstructionCache] = {}
    model_activations: dict[str, torch.Tensor] = {}

    # this hook just track the SAE input, output, features, and error. If `track_grads=True`, it also ensures
    # that requires_grad is set to True and retain_grad is called for intermediate values.
    def reconstruction_hook(sae_in: torch.Tensor, hook: HookPoint, hook_point: str):  # noqa: ARG001
        sae = saes[hook_point]
        feature_acts = sae.encode(sae_in)
        sae_out = sae.decode(feature_acts)
        sae_error = (sae_in - sae_out).detach().clone()
        if track_grads:
            track_grad(sae_error)
            track_grad(sae_out)
            track_grad(feature_acts)
            track_grad(sae_in)
        sae_activations[hook_point] = SaeReconstructionCache(
            sae_in=sae_in,
            feature_acts=feature_acts,
            sae_out=sae_out,
            sae_error=sae_error,
        )

        if include_error_term:
            return sae_out + sae_error
        return sae_out

    def sae_bwd_hook(output_grads: torch.Tensor, hook: HookPoint):  # noqa: ARG001
        # this just passes the output grads to the input, so the SAE gets the same grads despite the error term hackery
        return (output_grads,)

    # this hook just records model activations, and ensures that intermediate activations have gradient tracking turned on if needed
    def tracking_hook(hook_input: torch.Tensor, hook: HookPoint, hook_point: str):  # noqa: ARG001
        model_activations[hook_point] = hook_input
        if track_grads:
            track_grad(hook_input)
        return hook_input

    for hook_point in saes.keys():
        fwd_hooks.append(
            (hook_point, partial(reconstruction_hook, hook_point=hook_point))
        )
        bwd_hooks.append((hook_point, sae_bwd_hook))
    for hook_point in track_model_hooks or []:
        fwd_hooks.append((hook_point, partial(tracking_hook, hook_point=hook_point)))

    # now, just run the model while applying the hooks
    with model.hooks(fwd_hooks=fwd_hooks, bwd_hooks=bwd_hooks):
        model_output = model(input, return_type=return_type)

    return ApplySaesAndRunOutput(
        model_output=model_output,
        model_activations=model_activations,
        sae_activations=sae_activations,
    )

In [58]:
from dataclasses import dataclass
from transformer_lens.hook_points import HookPoint
from dataclasses import dataclass
from functools import partial
from typing import Any, Literal, NamedTuple

import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer
from transformer_lens.hook_points import HookPoint

EPS = 1e-8

torch.set_grad_enabled(True)


@dataclass
class AttributionGrads:
    metric: torch.Tensor
    model_output: torch.Tensor
    model_activations: dict[str, torch.Tensor]
    sae_activations: dict[str, SaeReconstructionCache]


@dataclass
class Attribution:
    model_attributions: dict[str, torch.Tensor]
    model_activations: dict[str, torch.Tensor]
    model_grads: dict[str, torch.Tensor]
    sae_feature_attributions: dict[str, torch.Tensor]
    sae_feature_activations: dict[str, torch.Tensor]
    sae_feature_grads: dict[str, torch.Tensor]
    sae_errors_attribution_proportion: dict[str, float]


def calculate_attribution_grads(
    model: HookedSAETransformer,
    prompt: str,
    metric_fn: Callable[[torch.Tensor], torch.Tensor],
    track_hook_points: list[str] | None = None,
    include_saes: dict[str, SAE] | None = None,
    return_logits: bool = True,
    include_error_term: bool = True,
) -> AttributionGrads:
    """
    Wrapper around apply_saes_and_run that calculates gradients wrt to the metric_fn.
    Tracks grads for both SAE feature and model neurons, and returns them in a structured format.
    """
    output = apply_saes_and_run(
        model,
        saes=include_saes or {},
        input=prompt,
        return_type="logits" if return_logits else "loss",
        track_model_hooks=track_hook_points,
        include_error_term=include_error_term,
        track_grads=True,
    )
    metric = metric_fn(output.model_output)
    output.zero_grad()
    metric.backward()
    return AttributionGrads(
        metric=metric,
        model_output=output.model_output,
        model_activations=output.model_activations,
        sae_activations=output.sae_activations,
    )


def calculate_feature_attribution(
    model: HookedSAETransformer,
    input: Any,
    metric_fn: Callable[[torch.Tensor], torch.Tensor],
    track_hook_points: list[str] | None = None,
    include_saes: dict[str, SAE] | None = None,
    return_logits: bool = True,
    include_error_term: bool = True,
) -> Attribution:
    """
    Calculate feature attribution for SAE features and model neurons following
    the procedure in https://transformer-circuits.pub/2024/march-update/index.html#feature-heads.
    This include the SAE error term by default, so inserting the SAE into the calculation is
    guaranteed to not affect the model output. This can be disabled by setting `include_error_term=False`.

    Args:
        model: The model to calculate feature attribution for.
        input: The input to the model.
        metric_fn: A function that takes the model output and returns a scalar metric.
        track_hook_points: A list of model hook points to track activations for, if desired
        include_saes: A dictionary of SAEs to include in the calculation. The key is the hook point to apply the SAE to.
        return_logits: Whether to return the model logits or loss. This is passed to TLens, so should match whatever the metric_fn expects (probably logits)
        include_error_term: Whether to include the SAE error term in the calculation. This is recommended, as it ensures that the SAE will not affecting the model output.
    """
    # first, calculate gradients wrt to the metric_fn.
    # these will be multiplied with the activation values to get the attributions
    outputs_with_grads = calculate_attribution_grads(
        model,
        input,
        metric_fn,
        track_hook_points,
        include_saes=include_saes,
        return_logits=return_logits,
        include_error_term=include_error_term,
    )
    model_attributions = {}
    model_activations = {}
    model_grads = {}
    sae_feature_attributions = {}
    sae_feature_activations = {}
    sae_feature_grads = {}
    sae_error_proportions = {}
    # this code is long, but all it's doing is multiplying the grads by the activations
    # and recording grads, acts, and attributions in dictionaries to return to the user
    with torch.no_grad():
        for name, act in outputs_with_grads.model_activations.items():
            assert act.grad is not None
            raw_activation = act.detach().clone()
            model_attributions[name] = (act.grad * raw_activation).detach().clone()
            model_activations[name] = raw_activation
            model_grads[name] = act.grad.detach().clone()
        for name, act in outputs_with_grads.sae_activations.items():
            assert act.feature_acts.grad is not None
            assert act.sae_out.grad is not None
            raw_activation = act.feature_acts.detach().clone()
            sae_feature_attributions[name] = (
                (act.feature_acts.grad * raw_activation).detach().clone()
            )
            sae_feature_activations[name] = raw_activation
            sae_feature_grads[name] = act.feature_acts.grad.detach().clone()
            if include_error_term:
                assert act.sae_error.grad is not None
                error_grad_norm = act.sae_error.grad.norm().item()
            else:
                error_grad_norm = 0
            sae_out_norm = act.sae_out.grad.norm().item()
            sae_error_proportions[name] = error_grad_norm / (
                sae_out_norm + error_grad_norm + EPS
            )
        return Attribution(
            model_attributions=model_attributions,
            model_activations=model_activations,
            model_grads=model_grads,
            sae_feature_attributions=sae_feature_attributions,
            sae_feature_activations=sae_feature_activations,
            sae_feature_grads=sae_feature_grads,
            sae_errors_attribution_proportion=sae_error_proportions,
        )


# prompt = " Tiger Woods plays the sport of"
# pos_token = model.tokenizer.encode(" golf")[0]
prompt = "In the beginning, God created the heavens and the"
pos_token = model.tokenizer.encode(" earth")
neg_token = model.tokenizer.encode(" sky")


def metric_fn(
    logits: torch.tensor,
    pos_token: torch.tensor = pos_token,
    neg_token: torch.Tensor = neg_token,
) -> torch.Tensor:
    return logits[0, -1, pos_token] - logits[0, -1, neg_token]


feature_attribution_df = calculate_feature_attribution(
    input=prompt,
    model=model,
    metric_fn=metric_fn,
    include_saes={sae.cfg.hook_name: sae},
    include_error_term=True,
    return_logits=True,
)

In [59]:
from transformer_lens.utils import test_prompt

test_prompt(prompt, model.to_string(pos_token), model)

Tokenized prompt: ['<|endoftext|>', 'In', ' the', ' beginning', ',', ' God', ' created', ' the', ' heavens', ' and', ' the']
Tokenized answer: [' earth']


Performance on answer token:
Rank: 0        Logit: 27.64 Prob: 99.32% Token: | earth|

Top 0th token. Logit: 27.64 Prob: 99.32% Token: | earth|
Top 1th token. Logit: 22.46 Prob:  0.56% Token: | Earth|
Top 2th token. Logit: 19.20 Prob:  0.02% Token: | planets|
Top 3th token. Logit: 18.80 Prob:  0.01% Token: | moon|
Top 4th token. Logit: 18.07 Prob:  0.01% Token: | heavens|
Top 5th token. Logit: 17.67 Prob:  0.00% Token: | oceans|
Top 6th token. Logit: 17.43 Prob:  0.00% Token: | ten|
Top 7th token. Logit: 17.41 Prob:  0.00% Token: | stars|
Top 8th token. Logit: 17.38 Prob:  0.00% Token: | seas|
Top 9th token. Logit: 17.35 Prob:  0.00% Token: | four|


Ranks of the answer tokens: [(' earth', 0)]

In [60]:
tokens = model.to_str_tokens(prompt)
unique_tokens = [f"{i}/{t}" for i, t in enumerate(tokens)]

px.bar(
    x=unique_tokens,
    y=feature_attribution_df.sae_feature_attributions[sae.cfg.hook_name][0]
    .sum(-1)
    .detach()
    .cpu()
    .numpy(),
)

In [61]:
def convert_sparse_feature_to_long_df(sparse_tensor: torch.Tensor) -> pd.DataFrame:
    """
    Convert a sparse tensor to a long format pandas DataFrame.
    """
    df = pd.DataFrame(sparse_tensor.detach().cpu().numpy())
    df_long = df.melt(ignore_index=False, var_name="column", value_name="value")
    df_long.columns = ["feature", "attribution"]
    df_long_nonzero = df_long[df_long["attribution"] != 0]
    df_long_nonzero = df_long_nonzero.reset_index().rename(
        columns={"index": "position"}
    )
    return df_long_nonzero


df_long_nonzero = convert_sparse_feature_to_long_df(
    feature_attribution_df.sae_feature_attributions[sae.cfg.hook_name][0]
)
df_long_nonzero.sort_values("attribution", ascending=False)

,position,feature,attribution
522,8,22597,2.092223
45,8,1941,0.839364
383,10,16873,0.657706
201,8,10531,0.602765
194,10,10195,0.570648
...,...,...,...
110,8,5013,-0.348029
234,10,11606,-0.410736
397,10,17519,-0.454039
307,10,14225,-0.473392


In [62]:
for i, v in (
    df_long_nonzero.query("position==8")
    .groupby("feature")
    .attribution.sum()
    .sort_values(ascending=False)
    .head(5)
    .items()
):
    print(f"Feature {i} had a total attribution of {v:.2f}")
    html = get_dashboard_html(
        sae_release="gpt2-small",
        sae_id=f"{sae.cfg.hook_layer}-res-jb",
        feature_idx=int(i),
    )
    display(IFrame(html, width=1200, height=300))

Feature 22597 had a total attribution of 2.09


Feature 1941 had a total attribution of 0.84


Feature 10531 had a total attribution of 0.60


Feature 1449 had a total attribution of 0.50


Feature 22638 had a total attribution of 0.49


In [63]:
for i, v in (
    df_long_nonzero.groupby("feature")
    .attribution.sum()
    .sort_values(ascending=False)
    .head(5)
    .items()
):
    print(f"Feature {i} had a total attribution of {v:.2f}")
    html = get_dashboard_html(
        sae_release="gpt2-small",
        sae_id=f"{sae.cfg.hook_layer}-res-jb",
        feature_idx=int(i),
    )
    display(IFrame(html, width=1200, height=300))

Feature 22597 had a total attribution of 2.09


Feature 16873 had a total attribution of 1.30


Feature 1449 had a total attribution of 1.22


Feature 1941 had a total attribution of 0.84


Feature 10195 had a total attribution of 0.62


# Advanced: Making U-Maps